In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local") \
    .appName("Exercise1") \
    .getOrCreate()

In [2]:
pwd

'/home/jovyan'

In [36]:
df = spark.read.csv("exampleData.csv", header=True, inferSchema=True)
df.show(1,vertical=True)

-RECORD 0--------------------------------
 _c0                 | 14                
 dateTime            | 01-jan-1990 00:00 
 indicator_rain      | 0                 
 precipitation       | 0.3               
 indicator_temp      | 0                 
 air_temperature     | 9.1               
 indicator_wetb      | 0                 
 wetb                | 9.0               
 dewpt               | 8.9               
 vappr               | 11.4              
 relative_humidity   | 99                
 msl                 | 1006.7            
 indicator_wdsp      | 2                 
 wind_speed          | 7                 
 indicator_wddir     | 2                 
 wind_from_direction | 190               
only showing top 1 row



In [6]:
df.createOrReplaceTempView("weatherConditions") 
staticSchema = df.schema

In [7]:
from pyspark.sql import functions as F 
df.filter((F.col('air_temperature')<5))

DataFrame[_c0: int, dateTime: string, indicator_rain: int, precipitation: string, indicator_temp: int, air_temperature: string, indicator_wetb: int, wetb: string, dewpt: string, vappr: string, relative_humidity: string, msl: string, indicator_wdsp: int, wind_speed: string, indicator_wddir: int, wind_from_direction: int]

In [37]:
streamingDF = spark.readStream \
    .schema(staticSchema) \
    .option("maxFilesPerTrigger", 1) \
    .format("csv") \
    .option("header", "true") \
    .load("exampleData.csv")

In [38]:
streamingDF.isStreaming

True

In [43]:
# # Write the filtered stream to a memory sink
# query = filteredDF.writeStream \
#     .format("memory") \
#     .queryName("temperature_stream") \
#     .outputMode("append") \
#     .start()

# # Wait for some data to be processed
# time.sleep(10)

# # Query the in-memory table
# spark.sql("SELECT * FROM temperature_stream").show(5)

# # Stop the query when done
# query.stop()

+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+
|_c0|dateTime|indicator_rain|precipitation|indicator_temp|air_temperature|indicator_wetb|wetb|dewpt|vappr|relative_humidity|msl|indicator_wdsp|wind_speed|indicator_wddir|wind_from_direction|
+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+
+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+



## Q: Try to read a response from API or json file or try simple scraping

In [13]:
import requests
import json
import time
import os

# Directory to save the JSON files
output_dir = "./streaming_data" 
os.makedirs(output_dir, exist_ok=True)

# Fetch the JSON data from the URL
url = "https://microsoftedge.github.io/Demos/json-dummy-data/512KB.json"
response = requests.get(url)
data = response.json()  # This is a list of JSON objects

# Limit to the first 50 records
data = data[:50]

# Simulate streaming by writing chunks of data to files
chunk_size = 10  # Number of records per file
for i in range(0, len(data), chunk_size):
    chunk = data[i:i + chunk_size]
    file_path = f"{output_dir}/data_{int(time.time())}_{i}.json"
    with open(file_path, "w") as f:
        json.dump(chunk, f)
    print(f"Wrote {len(chunk)} records to {file_path}")
    #time.sleep(1)  # Simulate delay between "arrivals" of new data

Wrote 10 records to ./streaming_data/data_1743169245_0.json
Wrote 10 records to ./streaming_data/data_1743169245_10.json
Wrote 10 records to ./streaming_data/data_1743169245_20.json
Wrote 10 records to ./streaming_data/data_1743169245_30.json
Wrote 10 records to ./streaming_data/data_1743169245_40.json


In [35]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the schema based on the JSON structure
json_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("country", StringType(), True)
])

# Read the stream
streamingDF_url = spark.readStream \
    .schema(json_schema) \
    .option("maxFilesPerTrigger", 1) \
    .format("json") \
    .load("./streaming_data/*.json")

# Check if it’s streaming
print("Is the DataFrame streaming?", streamingDF_url.isStreaming)

Is the DataFrame streaming? True


In [22]:
# note the format here
streamingDF_url = spark.readStream \
    .schema(json_schema) \
    .option("maxFilesPerTrigger", 1) \
    .format("json") \
    .load("./streaming_data/*.json")

In [23]:
print(streamingDF_url)
# streamingDF_url.show()

DataFrame[id: int, name: string, email: string, phone: string, address: string, city: string, state: string, zip: string, country: string]


#### Q: Try to view the results using print() or show(). See what you get

`print(streamingDF_url)` outputs the object representation, confirming it’s a streaming DataFrame but not showing the data.

`streamingDF_url.show()` raises an AnalysisException with the message: "Queries with streaming sources must be executed with writeStream.start(); FileSource[./streaming_data/*.json]".

Using `print()` displays the streaming DataFrame object but not the data. Calling `show()` fails with an AnalysisException because streaming DataFrames require a writeStream operation to process and display data.

#### Q: How can we know if the stream has loaded the data?

We can confirm the stream has loaded the data by checking `streamingDF.isStreaming`, which returns `True` if the DataFrame is a streaming source.

In [34]:
print("Is the DataFrame streaming?", streamingDF_url.isStreaming)

Is the DataFrame streaming? True


In [26]:
# Write the stream to a memory sink. To actually view the data, I need to write the stream to a sink
query = streamingDF_url.writeStream \
    .format("memory") \
    .queryName("json_stream") \
    .outputMode("append") \
    .start()

# Wait for some data to be processed
time.sleep(10)

# Query the in-memory table
spark.sql("SELECT * FROM json_stream").show(5)

# Stop the query when done
query.stop()

+----+-------------+-----+-----+-------+----+-----+----+-------+
|  id|         name|email|phone|address|city|state| zip|country|
+----+-------------+-----+-----+-------+----+-----+----+-------+
|NULL|Adeel Solangi| NULL| NULL|   NULL|NULL| NULL|NULL|   NULL|
|NULL|Afzal Ghaffar| NULL| NULL|   NULL|NULL| NULL|NULL|   NULL|
|NULL|Aamir Solangi| NULL| NULL|   NULL|NULL| NULL|NULL|   NULL|
|NULL|Abla Dilmurat| NULL| NULL|   NULL|NULL| NULL|NULL|   NULL|
|NULL|     Adil Eli| NULL| NULL|   NULL|NULL| NULL|NULL|   NULL|
+----+-------------+-----+-----+-------+----+-----+----+-------+
only showing top 5 rows



In [28]:
# Write the filtered stream to a memory sink
query = filteredDF.writeStream \
    .format("memory") \
    .queryName("filtered_json_stream") \
    .outputMode("append") \
    .start()

# Wait for some data to be processed
time.sleep(10)

# Query the in-memory table
spark.sql("SELECT * FROM filtered_json_stream").show(5)

# Stop the query when done
query.stop()

+---+----+-----+-----+-------+----+-----+---+-------+
| id|name|email|phone|address|city|state|zip|country|
+---+----+-----+-----+-------+----+-----+---+-------+
+---+----+-----+-----+-------+----+-----+---+-------+



### 8) Applying basic filtering to the stream (similar to exercise 1)

##### Back to using Example Data

In [39]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

StructType([
    StructField("_c0", IntegerType(), True),
    StructField("dateTime", StringType(), True),
    StructField("indicator_rain", IntegerType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("indicator_temp", IntegerType(), True),
    StructField("air_temperature", DoubleType(), True),
    StructField("indicator_wetb", IntegerType(), True),
    StructField("wetb", DoubleType(), True),
    StructField("dewpt", DoubleType(), True),
    StructField("vappr", DoubleType(), True),
    StructField("relative_humidity", IntegerType(), True),
    StructField("msl", DoubleType(), True),
    StructField("indicator_wdsp", IntegerType(), True),
    StructField("wind_speed", IntegerType(), True),
    StructField("indicator_wddir", IntegerType(), True),
    StructField("wind_from_direction", IntegerType(), True)
])

StructType([StructField('_c0', IntegerType(), True), StructField('dateTime', StringType(), True), StructField('indicator_rain', IntegerType(), True), StructField('precipitation', DoubleType(), True), StructField('indicator_temp', IntegerType(), True), StructField('air_temperature', DoubleType(), True), StructField('indicator_wetb', IntegerType(), True), StructField('wetb', DoubleType(), True), StructField('dewpt', DoubleType(), True), StructField('vappr', DoubleType(), True), StructField('relative_humidity', IntegerType(), True), StructField('msl', DoubleType(), True), StructField('indicator_wdsp', IntegerType(), True), StructField('wind_speed', IntegerType(), True), StructField('indicator_wddir', IntegerType(), True), StructField('wind_from_direction', IntegerType(), True)])

In [47]:
from pyspark.sql.functions import col

# Apply the filter for air_temperature < 5
filteredDF = streamingDF.filter(col("air_temperature") < 30)

I wrote the filtered stream to a memory sink using `filteredDF.writeStream`.

`format('memory')`.

`queryName('weatherQuery')`

`outputMode('append').start()`. 

The sink is memory, the output mode is append, and the trigger is the default unspecified mode (micro-batch with immediate processing). 

In [48]:
# Write the filtered stream to a memory sink
query = filteredDF.writeStream \
    .format("memory") \
    .queryName("weatherQuery") \
    .outputMode("append") \
    .start()

# Wait for some data to be processed (instead of awaitTermination)
time.sleep(10)

# Query the in-memory table to verify
spark.sql("SELECT * FROM weatherQuery").show(5)

# Stop the query when done
query.stop()

+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+
|_c0|dateTime|indicator_rain|precipitation|indicator_temp|air_temperature|indicator_wetb|wetb|dewpt|vappr|relative_humidity|msl|indicator_wdsp|wind_speed|indicator_wddir|wind_from_direction|
+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+
+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+



### 9) Writing Stream to a sink

##### Experimenting with another sink, mode and trigger 

Console Sink: Change `.format("memory")` to `.format("console")` to print to the console.

Update Mode: Change `.outputMode("append")` to `.outputMode("update")` (though it’s similar here without updates).

Fixed Trigger: Add `.trigger(processingTime="5 seconds")` to process every 5 seconds.

In [51]:
from pyspark.sql.functions import col
import time

# Assuming streamingDF and filteredDF are already defined
# If not, here’s the setup:
df = spark.read.csv("exampleData.csv", header=True, inferSchema=True)
staticSchema = df.schema
streamingDF = spark.readStream \
    .schema(staticSchema) \
    .option("maxFilesPerTrigger", 1) \
    .format("csv") \
    .option("header", "true") \
    .load("./exampleData/*.csv")  # Adjust path as needed
filteredDF = streamingDF.filter(col("air_temperature") < 10)

# Write the filtered stream to a console sink with update mode and 5-second trigger
query = filteredDF.writeStream \
    .format("console") \
    .outputMode("update") \
    .trigger(processingTime="5 seconds") \
    .start()

# Wait for some data to be processed
time.sleep(10)

# Since console sink is used, no SQL query is possible; output goes to console
# Stop the query when done
query.stop()

The default setup (memory sink, append mode, unspecified trigger) and the suggested modifications (console sink, update mode, 5-second trigger) all process the same filtered data (air_temperature < 10) from exampleData.csv as a stream. 

They share micro-batch processing and output the same rows, differing only in where the data goes (memory vs. console), how it’s written (append vs. update, though similar here), and when it’s processed (immediate vs. every 5 seconds). The practical output remains consistent with a single file and no updates.

#### 10) We can query the data while stream is being written and processed

In [52]:
from pyspark.sql.functions import col
import time

# Setup from previous steps
df = spark.read.csv("exampleData.csv", header=True, inferSchema=True)
staticSchema = df.schema
streamingDF = spark.readStream \
    .schema(staticSchema) \
    .option("maxFilesPerTrigger", 1) \
    .format("csv") \
    .option("header", "true") \
    .load("./exampleData/*.csv")  # Adjust path as needed
filteredDF = streamingDF.filter(col("air_temperature") < 10)

# Write the filtered stream to a memory sink, since we wrote it to a console in the last step 
query = filteredDF.writeStream \
    .format("memory") \
    .queryName("weatherQuery") \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .start()

# Wait for some data to be processed
time.sleep(15)  # Wait for at least 3 triggers (15s > 3 * 5s)

# Query the in-memory table
spark.sql("SELECT * FROM weatherQuery").show(5)

# Stop the query
query.stop()

+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+
|_c0|dateTime|indicator_rain|precipitation|indicator_temp|air_temperature|indicator_wetb|wetb|dewpt|vappr|relative_humidity|msl|indicator_wdsp|wind_speed|indicator_wddir|wind_from_direction|
+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+
+---+--------+--------------+-------------+--------------+---------------+--------------+----+-----+-----+-----------------+---+--------------+----------+---------------+-------------------+



#### Q: Is there any difference between methods and operations provided for structured streaming using sparksession and spark streamingcontext?

Yes, there are differences between methods and operations for Structured Streaming using `SparkSession` and the older Spark Streaming using `StreamingContext`. 

`SparkSession` offers a high-level DataFrame API with `.readStream()` and `.writeStream()`, supporting SQL queries, event-time processing, and continuous mode, as seen in this exercise. 

`StreamingContext` uses a lower-level DStream API based on RDDs, with operations like `map` and `reduce`, lacking advanced features and DataFrame integration. 

`SparkSession` is more modern and unified with batch processing.

I queried the stream using `spark.sql('SELECT * FROM weatherQuery').show(5)` after writing it to a `memory` sink with `queryName('weatherQuery')`. 

The difference between Structured Streaming with `SparkSession` and Spark Streaming with `StreamingContext` lies in their APIs: `SparkSession` provides a high-level DataFrame interface with advanced features like event-time and SQL, while `StreamingContext` uses a lower-level DStream RDD API without such capabilities.

###### Exercise 1

From working through the codes and questions in this Structured Streaming exercise, I have an understanding of Spark’s streaming capabilities. I have learned how to set up a streaming DataFrame using Spark.readStream, applying schemas like staticSchema to process data from files. By filtering streams (e.g., air_temperature < 10), I can manipulate streaming data with DataFrame operations. Writing streams to sinks (memory and console) has taught me about output destinations, while experimenting with append and update modes and triggers (e.g., 5-second intervals) highlighted control over data processing timing and behaviour. Querying in-memory tables with SQL has taught me how to inspect streaming results interactively. Troubleshooting errors (e.g., schema mismatches, syntax issues) likely deepened my problem-solving skills. Overall, I have learned about practical stream processing and configuration skills within Spark’s Structured Streaming framework.

In [53]:
# import time
# time.sleep(10)

Based on the Apache Spark Structured Streaming Programming Guide (https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html), watermarking is a mechanism used to handle late-arriving data in streaming applications, particularly when processing event-time-based operations like aggregations over time windows.

Watermarking in Apache Spark Structured Streaming defines a threshold for how late data can arrive and still be included in computations before it is considered too old to process. It is tied to the concept of event time—the time when an event actually occurred (e.g., a timestamp in your data), rather than the processing time. Late data arises in streaming systems due to delays in data generation, network issues, or out-of-order delivery.

A watermark is essentially a moving boundary in event time. It specifies how much "lateness" Spark tolerates before discarding old data from its state (e.g., aggregations). Once the watermark advances past a certain point, data with event times older than the watermark is dropped, allowing Spark to manage memory efficiently by clearing outdated state.

Watermarking ensures that Spark can produce correct, up-to-date results for time-based operations (like windowed aggregations) while handling late data without keeping infinite state. 

Watermarking is implemented in Structured Streaming via the `.withWatermark()` method on a DataFrame, typically before performing aggregations or joins that rely on event time. There is an even-time column and a delay threshold. For example, the column containing the timestamp of when events occurred (e.g., `timestamp` in exampleData.csv if parsed as a `TimestampType`). 

The code could be written as 'streamingDF = streamingDF.withWatermark("dateTime", "10 minutes")` Any data with an event time older than this watermark is ignored in subsequent computations. 

Watermarking is most commonly used with windowed aggregations, where you group data into time windows based on event time. For example: 

`from` pyspark.sql.functions `import` window, col

Windowed count of records per 5-minute interval, with watermark
`aggDF = streamingDF \
    .withWatermark("dateTime", "10 minutes") \
    .groupBy(window("dateTime", "5 minutes")) \
    .count()`

The `window()` function creates 5-minute buckets based on `dateTime`.
The watermark of "10 minutes" means Spark keeps state for each window for up to 10 minutes after the window ends. If a late event arrives within this 10-minute grace period, it updates the count; otherwise, it’s discarded.
Once the watermark advances (based on the latest event time seen), Spark drops state for windows older than `watermark - 10 minutes`

**Watermarking interacts with output modes**:

**Append Mode:** Only finalized windows (those older than the watermark) are output. Late data within the watermark updates the window until it’s finalized.
**Update Mode:** Outputs changes to windows as late data arrives, up until the watermark expires the window.

For example: 
`query = aggDF.writeStream \
    .format("console") \
    .outputMode("update") \
    .trigger(processingTime="1 minute") \
    .start()`
The output mode here is "update"